<a href="https://colab.research.google.com/github/marthab-oss/Machine-Learning-Course-2026/blob/main/DeepRL_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforcement Learning with TorchRL — Homework

This notebook is a **homework version** of the TorchRL tutorial.

## How to use this notebook
- **Your job** is to fill in the code in cells marked **TODO**.
- **Do not change boilerplate** (imports, env setup, plotting). Focus on the RL *flow*:
  - build behavior policy (Q-values → action + exploration)
  - collect experience
  - store in replay buffer
  - sample batches and optimize the RL objective
  - update target network
  - evaluate periodically (deterministic)
- **Solutions are included** right after each TODO cell in a collapsed cell titled **SOLUTION**. Only expand if you are completely stuck.

> Based on the official [TorchRL documentation](https://docs.pytorch.org/rl/stable/index.html).

In [ ]:
# Install dependencies only if running in Google Colab
import sys

if "google.colab" in sys.modules:
    !pip install torchrl tensordict gymnasium[classic-control] tqdm matplotlib -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.8/536.8 kB 46.2 MB/s eta 0:00:00


## What is Reinforcement Learning?

In RL, an **agent** learns by interacting with an **environment**.

At each step:
1. The agent receives an observation (state).
2. The agent chooses an action.
3. The environment returns a reward and next observation.
4. The agent updates its behavior using collected experience.

```
┌───────────┐   action    ┌─────────────┐
│           │────────────▶│             │
│   Agent   │             │ Environment │
│  (Policy) │◀────────────│  (CartPole) │
│           │ observation │             │
└───────────┘   reward    └─────────────┘
```

The objective is to maximize cumulative reward over time.

### CartPole
<img src="https://github.com/raphaelschwinger/Introduction-to-Deep-Reinforcement-Learning/blob/main/figures/cart_pole.gif?raw=1" alt="CartPole demo" width="400"/>



- **Observation:** 4 numbers (cart position/velocity, pole angle/angular velocity).
- **Action:** left (`0`) or right (`1`).
- **Reward:** `+1` each step the pole stays upright.
- **Done:** pole falls too far, cart leaves range, or 500 steps reached.

## Setup

In [ ]:

from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from torch import nn
from tensordict import TensorDict
from tensordict.nn import TensorDictModule, TensorDictSequential
from tqdm import tqdm

from torchrl.collectors import SyncDataCollector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import Compose, ExplorationType, StepCounter, TransformedEnv, set_exploration_type
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs
from torchrl.modules import EGreedyModule, QValueActor
from torchrl.objectives import DQNLoss, SoftUpdate

%matplotlib inline

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
torch.manual_seed(42)

Using device: cuda


## TensorDict: The Data Backbone

TorchRL uses `TensorDict` to pass RL data through environments, policies, collectors, and losses.

A `TensorDict` is like a dictionary of tensors with a shared batch shape. In RL, this is ideal because transitions naturally come as groups: observation, action, reward, done, and next observation.

In [ ]:
batch_size = 4

data = TensorDict(
    {
        "obs": torch.randn(batch_size, 3),
        "action": torch.randint(0, 2, (batch_size, 1)),
        "reward": torch.randn(batch_size, 1),
    },
    batch_size=[batch_size],
)

print(data)

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.int64, is_shared=False),
        obs: Tensor(shape=torch.Size([4, 3]), device=cpu, dtype=torch.float32, is_shared=False),
        reward: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([4]),
    device=None,
    is_shared=False)


You can index, move, and stack `TensorDict`s while preserving the key structure.

In [ ]:
print("First element:\n", data[0])

data_on_device = data.to(device)
print("\nMoved to:", data_on_device.device)

stacked = torch.stack([data, data], dim=0)
print("\nStacked batch size:", stacked.batch_size)

First element:
 TensorDict(
    fields={
        action: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        obs: Tensor(shape=torch.Size([3]), device=cpu, dtype=torch.float32, is_shared=False),
        reward: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)

Moved to: cuda:0

Stacked batch size: torch.Size([2, 4])


`TensorDict` can also be nested. TorchRL commonly stores next-step values under a `"next"` nested key.

In [ ]:
nested_td = TensorDict(
    {
        "observation": TensorDict(
            {
                "vector": torch.randn(2, 4),
                "pixels": torch.randn(2, 3, 32, 32),
            },
            batch_size=[2],
        ),
        "next": TensorDict(
            {
                "reward": torch.randn(2, 1),
                "done": torch.zeros(2, 1, dtype=torch.bool),
            },
            batch_size=[2],
        ),
    },
    batch_size=[2],
)

print(nested_td)

TensorDict(
    fields={
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([2, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                reward: Tensor(shape=torch.Size([2, 1]), device=cpu, dtype=torch.float32, is_shared=False)},
            batch_size=torch.Size([2]),
            device=None,
            is_shared=False),
        observation: TensorDict(
            fields={
                pixels: Tensor(shape=torch.Size([2, 3, 32, 32]), device=cpu, dtype=torch.float32, is_shared=False),
                vector: Tensor(shape=torch.Size([2, 4]), device=cpu, dtype=torch.float32, is_shared=False)},
            batch_size=torch.Size([2]),
            device=None,
            is_shared=False)},
    batch_size=torch.Size([2]),
    device=None,
    is_shared=False)


## The Environment

TorchRL wraps Gymnasium environments and returns/accepts `TensorDict`s directly.

For DQN, we use `categorical_action_encoding=True` so actions are integer categories (`0` or `1`) compatible with `QValueActor`.

In [ ]:
env = TransformedEnv(
    GymEnv("CartPole-v1", categorical_action_encoding=True, device="cpu"),
    Compose(StepCounter()),
)

check_env_specs(env)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2026-04-28 15:32:59,568 [torchrl][INFO]    check_env_specs succeeded! [END]


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
print("Observation spec:", env.observation_spec)
print("Action spec:", env.action_spec)
print("Reward spec:", env.reward_spec)

Observation spec: Composite(
    observation: BoundedContinuous(
        shape=torch.Size([4]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, contiguous=True)),
        device=cpu,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=cpu,
    shape=torch.Size([]),
    data_cls=None)
Action spec: Categorical(
    shape=torch.Size([]),
    space=CategoricalBox(n=2),
    device=cpu,
    dtype=torch.int64,
    domain=discrete)
Reward spec: UnboundedContinuous(
    sh

### Interacting with the environment

In [ ]:
td = env.reset()
print(td)

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=cpu,
    is_shared=False)


In [ ]:
td_step = env.rand_step(td)
print(td_step)

TensorDict(
    fields={
        action: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.int64, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([]),
            device=cpu,
            is_shared=False),
        observa

In [ ]:
random_rollout = env.rollout(max_steps=50)
print("Rollout batch size:", random_rollout.batch_size)
print("Keys:", random_rollout.keys())
print("Rollout reward:", random_rollout["next"]["reward"].sum().item())

Rollout batch size: torch.Size([14])
Keys: _StringKeys(dict_keys(['observation', 'done', 'terminated', 'truncated', 'step_count', 'action', 'next']))
Rollout reward: 14.0


## Building the Agent

### `TensorDictModule` in one minute

A `TensorDictModule` wraps a regular PyTorch module and reads/writes named keys in a `TensorDict`.

In [ ]:
toy_module = TensorDictModule(
    nn.Linear(4, 2),
    in_keys=["observation"],
    out_keys=["logits"],
)

toy_td = TensorDict({"observation": torch.randn(3, 4)}, batch_size=[3])
print(toy_module(toy_td))

TensorDict(
    fields={
        logits: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.float32, is_shared=False),
        observation: Tensor(shape=torch.Size([3, 4]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([3]),
    device=None,
    is_shared=False)


### DQN policy with TorchRL

For this notebook, think of DQN as:
- a network that predicts one value per action (`Q(s, a)`),
- plus an exploration strategy (epsilon-greedy) during training.

We will use:
- `QValueActor` to turn Q-values into actions,
- `EGreedyModule` for exploration,
- `TensorDictSequential` to compose policy + exploration.

In [ ]:
# =========================
# Exercise 1 — Behavior policy
# =========================
# Goal: build a DQN *behavior* policy in TorchRL.
# - q_net produces action-values Q(s, a) for each discrete action.
# - policy selects the greedy action (argmax_a Q(s, a)).
# - exploration_policy injects epsilon-greedy exploration during training.
# - behavior_policy composes greedy policy + exploration.

max_steps = 500_000

# TODO: define `q_net` as a TensorDictModule mapping:
#   in_keys=["observation"] -> out_keys=["action_value"]
# and producing 2 action-values for CartPole.
q_net = TensorDictModule(
    nn.Sequential(
        nn.Linear(4,12),
        nn.ReLU(),
        nn.Linear(12,12),
        nn.ReLU(),
        nn.Linear(12,2)
    ),
    in_keys=('observation'), out_keys=('action_value')
  )

# TODO: wrap `q_net` into a QValueActor named `policy`.
policy = QValueActor(q_net, in_keys=["observation"], spec=env.action_spec)


# TODO: create an epsilon-greedy module named `exploration_policy`.
# Use eps_init=1.0, eps_end=0.001, annealing_num_steps=max_steps*0.5.
exploration_policy = EGreedyModule(spec=env.action_spec, eps_init=1.0, eps_end=0.001, annealing_num_steps=max_steps*0.5)

# TODO: compose the greedy policy and exploration into `behavior_policy`.
behavior_policy = TensorDictSequential(policy, exploration_policy)

# Quick sanity check (should run once TODOs are filled)
sample_td = env.reset()
print("Greedy policy output:")
print(policy(sample_td.clone()))
print("\nBehavior policy output (with exploration):")
print(behavior_policy(sample_td.clone()))



Greedy policy output:
TensorDict(
    fields={
        action: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.int64, is_shared=False),
        action_value: Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.float32, is_shared=False),
        chosen_action_value: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        observation: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=cpu,
    is_shared=False)

Behavior policy output (with exploration):
TensorDict(
    fields={
        action: Tens

### Solution

In [ ]:
# =========================
# SOLUTION — Exercise 1
# =========================

max_steps = 500_000

q_net = TensorDictModule(
    nn.Sequential(
        nn.Linear(4, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
        nn.ReLU(),
        nn.Linear(128, 2),
    ),
    in_keys=["observation"],
    out_keys=["action_value"],
)

policy = QValueActor(q_net, in_keys=["observation"], spec=env.action_spec)

exploration_policy = EGreedyModule(
    spec=env.action_spec,
    eps_init=1.0,
    eps_end=0.001,
    annealing_num_steps=max_steps * 0.5,
)

behavior_policy = TensorDictSequential(policy, exploration_policy)


## Data Collection

TorchRL separates interaction and optimization:
- A **collector** interacts with the environment and emits batches of transitions.
- A **replay buffer** stores these transitions.
- The optimizer samples random mini-batches from replay for training.

This is the core DQN data flow in TorchRL.

In [ ]:
# =========================
# Exercise 2 — Data collection + replay
# =========================
# Goal: instantiate a collector that interacts with the env using `behavior_policy`,
# and a replay buffer to store transitions.

# TODO: create `collector` with frames_per_batch=1000 and total_frames=max_steps.
collector = SyncDataCollector(create_env_fn=lambda, policy=behavior_policy, frames_per_batch=1000, total_frames=max_steps)

# TODO: create `replay_buffer` backed by LazyTensorStorage with max_size=100_000.
replay_buffer = LazyTensorStorage(max_size=100_000)


NameError: name 'Collector' is not defined

### Solution

In [ ]:
# =========================
# SOLUTION — Exercise 2
# =========================

collector = SyncDataCollector(
    create_env_fn=lambda: GymEnv("CartPole-v1", categorical_action_encoding=True, device="cpu"),
    policy=behavior_policy,
    frames_per_batch=1000,
    total_frames=max_steps,
)

replay_buffer = ReplayBuffer(storage=LazyTensorStorage(max_size=100_000))


In [ ]:
# =========================
# Exercise 3 — Objective + target network
# =========================
# Goal: connect the policy to a DQN objective and configure target-network updates.

# TODO: create the DQN loss module.
loss_module = ...

# TODO: configure the value estimator with gamma=0.99.
...

# TODO: create the target network updater with tau=0.001.
target_net_updater = ...

# TODO: create an Adam optimizer over policy parameters with lr=1e-4.
optimizer = ...

print("Collector, replay buffer, and DQN loss are ready.")


In [ ]:
# =========================
# SOLUTION — Exercise 3
# =========================

loss_module = DQNLoss(policy, action_space=env.action_spec)
loss_module.make_value_estimator(gamma=0.99)
target_net_updater = SoftUpdate(loss_module, tau=0.001)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-4)


## Training Loop with TorchRL

The loop below follows a reusable TorchRL pattern:
1. collect transitions,
2. store transitions,
3. sample mini-batches,
4. optimize loss,
5. evaluate periodically.

We keep algorithm details minimal and focus on **how components connect**.

In [ ]:
# =========================
# Exercise 4 — Training loop (TorchRL flow)
# =========================
# Fill the TODOs to implement the standard TorchRL DQN loop:
# collect -> store -> sample -> optimize -> target update -> evaluate.

logs = defaultdict(list)
total_frames = 0
min_buffer_size = 5_000
updates_per_batch = 20
batch_size = 256

def evaluate_policy(env, policy, n_episodes=5, max_steps=500):
    """Return mean episodic return under a deterministic (greedy) policy."""
    # TODO: run n_episodes rollouts with deterministic exploration type and return mean return.
    ...

pbar = tqdm(total=max_steps, desc="Training")

for i, batch in enumerate(collector):
    # TODO: store newly collected transitions in replay.
    ...

    frames = batch.numel()
    total_frames += frames
    pbar.update(frames)

    train_reward = batch["next", "reward"].mean().item()
    logs["train_reward"].append(train_reward)
    logs["frames"].append(total_frames)

    if len(replay_buffer) >= min_buffer_size:
        for _ in range(updates_per_batch):
            # TODO: sample a minibatch, compute loss, backprop, optimizer step.
            # Also update the target network each gradient step.
            ...

        # TODO: log the most recent scalar loss value.
        ...
    else:
        logs["loss"].append(float("nan"))

    # TODO: decay epsilon according to number of frames collected.
    ...
    logs["eps"].append(float(exploration_policy.eps))

    if i % 5 == 0:
        # TODO: compute evaluation return and log it.
        ...

        pbar.set_postfix(
            eval_reward=f"{eval_reward:.1f}",
            eps=f"{float(exploration_policy.eps):.3f}",
        )

pbar.close()
collector.shutdown()
print("Training complete.")


### Solution

In [ ]:
# =========================
# SOLUTION — Exercise 4
# =========================

logs = defaultdict(list)
total_frames = 0
min_buffer_size = 5_000
updates_per_batch = 20
batch_size = 256

pbar = tqdm(total=max_steps, desc="Training")

for i, batch in enumerate(collector):
    replay_buffer.extend(batch)
    frames = batch.numel()
    total_frames += frames
    pbar.update(frames)

    train_reward = batch["next", "reward"].mean().item()
    logs["train_reward"].append(train_reward)
    logs["frames"].append(total_frames)

    if len(replay_buffer) >= min_buffer_size:
        for _ in range(updates_per_batch):
            sample = replay_buffer.sample(batch_size)
            loss_td = loss_module(sample)
            loss = loss_td["loss"]

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            target_net_updater.step()

        logs["loss"].append(loss.item())
    else:
        logs["loss"].append(float("nan"))

    exploration_policy.step(frames=frames)
    logs["eps"].append(float(exploration_policy.eps))

    if i % 5 == 0:
        with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
            eval_rewards = []
            for _ in range(5):
                eval_td = env.rollout(max_steps=500, policy=policy)
                eval_rewards.append(eval_td["next", "reward"].sum().item())
            eval_reward = sum(eval_rewards) / len(eval_rewards)

        logs["eval_reward"].append(eval_reward)
        logs["eval_frames"].append(total_frames)

        pbar.set_postfix(
            eval_reward=f"{eval_reward:.1f}",
            eps=f"{float(exploration_policy.eps):.3f}",
        )

pbar.close()
collector.shutdown()
print("Training complete.")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 8))

axes[0, 0].plot(logs["frames"], logs["train_reward"])
axes[0, 0].set_title("Train Reward (per batch)")
axes[0, 0].set_xlabel("Frames")
axes[0, 0].set_ylabel("Mean reward")

axes[0, 1].plot(logs["eval_frames"], logs["eval_reward"])
axes[0, 1].set_title("Evaluation Return")
axes[0, 1].axhline(500, linestyle="--")
axes[0, 1].set_xlabel("Frames")
axes[0, 1].set_ylabel("Episode return")


axes[1, 0].plot(logs["frames"], logs["loss"])
axes[1, 0].set_title("Loss")
axes[1, 0].set_xlabel("Frames")
axes[1, 0].set_ylabel("Loss")

axes[1, 1].plot(logs["frames"], logs["eps"])
axes[1, 1].set_title("Exploration Epsilon")
axes[1, 1].set_xlabel("Frames")
axes[1, 1].set_ylabel("Epsilon")


plt.tight_layout()
plt.show()

## Visualize the Trained Agent

In [ ]:
from matplotlib import animation
from IPython.display import HTML

render_env = TransformedEnv(
    GymEnv("CartPole-v1", render_mode="rgb_array", categorical_action_encoding=True, device="cpu"),
    Compose(StepCounter()),
)

frames = []
td = render_env.reset()

with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
    for _ in range(500):
        td = policy(td)
        td = render_env.step(td)
        frames.append(render_env.base_env._env.render())
        if td["next", "done"].item():
            break
        td = td["next"]

render_env.close()
print(f"Episode lasted {len(frames)} steps")

fig, ax = plt.subplots(figsize=(5, 4))
ax.axis("off")
img = ax.imshow(frames[0])

def update(frame):
    img.set_data(frame)
    return [img]

ani = animation.FuncAnimation(fig, update, frames=frames, interval=30, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

## Summary

In this notebook, the focus was TorchRL workflow rather than DQN theory.

| Concept | TorchRL component |
|---|---|
| Environment | `GymEnv` + `TransformedEnv` |
| Data carrier | `TensorDict` |
| Policy | `QValueActor` |
| Exploration | `EGreedyModule` |
| Data collection | `SyncDataCollector` + `ReplayBuffer` |
| Training objective | `DQNLoss` + `SoftUpdate` |

**Key takeaways**
- RL training in TorchRL is a modular pipeline.
- `TensorDict` is the interface that lets modules compose cleanly.
- A practical loop is: collect -> store -> sample -> optimize -> evaluate.

**Next steps**
- Try changing `frames_per_batch`, `batch_size`, and `lr`.
- Swap CartPole for another Gymnasium environment.
- Explore more TorchRL components in the [official docs](https://docs.pytorch.org/rl/stable/index.html).